# Work Script: privacy_scan
Converted from `sft/work/privacy_scan.py`.


In [ ]:
"""
sft/work/privacy_scan.py
Privacy scan: take 200 random utterances >=40 chars from the source CSV and
assert that none appears as a substring in:
  - sft/notebooks/P1a_g1_e1_profile.ipynb
  - sft/reports/g1_report.json
  - sft/reports/e1_profile.json
Exit 0 = clean. Exit 1 = violation found.
"""
import json
import random
import sys
from pathlib import Path

REPO_ROOT = Path(__file__).resolve().parent.parent.parent  # Capstone/
CSV_PATH  = REPO_ROOT / "data" / "processed" / "integrated_conversations_ner_validated.csv"
NB_PATH   = REPO_ROOT / "sft" / "notebooks" / "P1a_g1_e1_profile.ipynb"
TARGETS   = [
    NB_PATH,
    REPO_ROOT / "sft" / "reports" / "g1_report.json",
    REPO_ROOT / "sft" / "reports" / "e1_profile.json",
]

SEED      = 42
N_SAMPLE  = 200
MIN_LEN   = 40

random.seed(SEED)

# ── 1. sample utterances ──────────────────────────────────────────────────────
import pandas as pd
df = pd.read_csv(
    CSV_PATH,
    usecols=["utterance"],
    dtype={"utterance": "object"},
    encoding="utf-8",
    low_memory=False,
)
candidates = (
    df["utterance"]
    .dropna()
    .astype(str)
    .pipe(lambda s: s[s.str.len() >= MIN_LEN])
    .tolist()
)
if len(candidates) < N_SAMPLE:
    print(f"WARNING: only {len(candidates)} candidates (need {N_SAMPLE})")
    sample = candidates
else:
    sample = random.sample(candidates, N_SAMPLE)

print(f"Sampled {len(sample)} utterances (min_len={MIN_LEN})")

# ── 2. load target file contents ──────────────────────────────────────────────
target_contents: dict[str, str] = {}
for p in TARGETS:
    if p.exists():
        target_contents[p.name] = p.read_text(encoding="utf-8")
        print(f"  Loaded target: {p.name}  ({len(target_contents[p.name]):,} chars)")
    else:
        print(f"  WARNING: target not found: {p}")

# ── 3. check each utterance ───────────────────────────────────────────────────
violations: list[dict] = []
for i, utt in enumerate(sample):
    # Use lowercase comparison to be conservative
    utt_lower = utt.lower()
    for fname, content in target_contents.items():
        if utt_lower in content.lower():
            violations.append({
                "utterance_index": i,
                "utterance_len": len(utt),
                "file": fname,
                "snippet": utt[:80],
            })

# ── 4. report ─────────────────────────────────────────────────────────────────
print()
if violations:
    print(f"PRIVACY SCAN FAILED — {len(violations)} violation(s):")
    for v in violations:
        print(f"  [{v['file']}] idx={v['utterance_index']} len={v['utterance_len']}")
        print(f"    snippet omitted (privacy)")
    sys.exit(1)
else:
    print(f"PRIVACY SCAN PASSED — 0 of {len(sample)} utterances found in any target file.")
    sys.exit(0)

In [ ]:
print("RESULT: PASS privacy_scan.py notebook compiled successfully")